In [7]:
pip install xlsxwriter

Note: you may need to restart the kernel to use updated packages.


In [86]:
# =====================================================
# Normalization / Splitting / Matching
# =====================================================

def normalize(s):
    if not isinstance(s, str):
        return ""
    s = re.sub(r"\([^)]*\)", "", s)     # remove (...)
#    s = re.sub(r"\b\d+\b", "", s)       # remove stand-alone numbers
    s = re.sub(r"[-‐-–—]", " ", s) 
    s = re.sub(r"^[^\w]+", "", s)
    s = re.sub(r"\s+", " ", s)          # collapse whitespace
    s = re.sub(r"[^\w]+$", "", s)
    return s.strip().lower()

def split_multi(s):
    if not isinstance(s, str) or not s.strip():
        return []
    return [normalize(x) for x in s.split(";") if normalize(x)]

def subset_match_list(gt_list, pred_list):
    if not gt_list or not pred_list:
        return False
    return any((g in p) or (p in g) for g in gt_list for p in pred_list)

def exact_match(gt, pred):
    return normalize(gt) == normalize(pred)

In [ ]:
from rapidfuzz import fuzz

def fuzzy_match(a, b, threshold=85):
    """Fuzzy match using token_set_ratio (handles word order, extra tokens)."""
    a = normalize(a)
    b = normalize(b)
    if not a or not b:
        return False
    return fuzz.token_set_ratio(a, b) >= threshold


In [ ]:
import pandas as pd

#excel_path = "D:/GeneAgent3/干净数据/20251112_ALL_LLM_results_with_metadata_full_name.xlsx"   # change this
excel_path = "D:/GeneAgent3/干净数据/20251112_ALL_LLM_results_with_metadata_cleaned.xlsx"
xls = pd.read_excel(excel_path, sheet_name=None)

# Store row indices where ALL predicted fields are NaN for each sheet
missing_all4_per_sheet = {}

def detect_pred_cols(df):
    """Detect the 4 predicted columns automatically."""
    pred_cols = []
    for col in df.columns:
        col_low = col.lower()
        if ("symbol (" in col_low or
            "gene name (" in col_low or
            "disease (" in col_low or
            "chd phenotype (" in col_low):
            pred_cols.append(col)
    return pred_cols


# -----------------------------------------------------------
# 1. Gather rows where ALL 4 predicted fields are NaN
# -----------------------------------------------------------

for sheet_name, df in xls.items():
    pred_cols = detect_pred_cols(df)

    # mask: row is True only if all 4 predicted columns are NaN
    mask = df[pred_cols].isna().all(axis=1)
    missing_rows = df.index[mask].tolist()

    missing_all4_per_sheet[sheet_name] = set(missing_rows)

    print(f"{sheet_name}: {len(missing_rows)} rows where ALL FOUR predicted fields are NaN")


# -----------------------------------------------------------
# 2. Intersection across ALL sheets
# -----------------------------------------------------------

all_sets = list(missing_all4_per_sheet.values())

#common_missing_rows = [380, 381, 382, 385, 389, 390, 391, 396, 397, 399, 400, 408, 411, 414, 415, 417, 420]

print("\n=== COMMON ROWS with ALL FOUR predicted fields NaN across ALL sheets ===")
print(all_sets)
#print("Count:", len(common_missing_rows))
#print("Row indices:", sorted(common_missing_rows))




In [ ]:
def aggregate_gene(df_group, model, gene_id):
    # ----------- GT: only from first incident row -----------
    first_idx = df_group.index.min()
    first     = df_group.loc[first_idx]

    gt_symbol        = first.get("symbol", "").strip()
    gt_gene_name     = first.get("gene_name", "").strip()
    gt_disease_raw   = first.get("disease", "")
    gt_phenotype_raw = first.get("phenotypes", "")

    # ----------- GT consistency check -----------
    inconsistencies = {}
    for col in ["symbol", "gene_name", "disease", "phenotypes"]:
        uniq = df_group[col].astype(str).unique()
        if len(uniq) > 1:
            inconsistencies[col] = list(uniq)

    # ----------- Aggregate prediction raw strings -----------
    pred_dis_raw  = "; ".join(df_group[dis_col].dropna().astype(str))
    pred_phe_raw  = "; ".join(df_group[phe_col].dropna().astype(str))
    pred_sym_raw  = "; ".join(df_group[sym_col].dropna().astype(str))
    pred_name_raw = "; ".join(df_group[name_col].dropna().astype(str))

    # ----------- Convert GT + Pred to lists -----------
    gt_dis_list   = split_multi(gt_disease_raw)
    gt_phe_list   = split_multi(gt_phenotype_raw)

    pred_dis_list = split_multi(pred_dis_raw)
    pred_phe_list = split_multi(pred_phe_raw)
    pred_sym_list = list({normalize(x) for x in pred_sym_raw.split(";") if normalize(x)})
    pred_name_list = list({normalize(x) for x in pred_name_raw.split(";") if normalize(x)})

    # ----------- Matching -----------

    # SYMBOL matching (exact only is recommended)
    symbol_match = any(exact_match(gt_symbol, p) for p in pred_sym_list)

    # GENE NAME MATCH (fuzzy + exact)
    gene_match = any(
        exact_match(gt_gene_name, p) or fuzzy_match(gt_gene_name, p, threshold=85)
        for p in pred_name_list
    )

    # Detect GT disease missing
    
    if normalize(gt_disease_raw) == "":
        disease_match = None  # or use np.nan
    else:
        # your existing matching logic:
        disease_match = (
            subset_match_list(gt_dis_list, pred_dis_list) or
            any(fuzzy_match(g, p, threshold=65) 
                for g in gt_dis_list for p in pred_dis_list)
        )


    # PHENOTYPE MATCH (exact subset + fuzzy)
    phenotype_match = (
        subset_match_list(gt_phe_list, pred_phe_list) or
        any(fuzzy_match(g, p, threshold=85) 
            for g in gt_phe_list for p in pred_phe_list)
    )

    # ----------- Return final aggregated record -----------
    return {
        "gene_id": gene_id,                          # <- FIXED
        "symbol": gt_symbol,                         # <- correct GT symbol
        "Pred_symbol_list": "; ".join(pred_sym_list),
        "symbol_match": symbol_match,

        "gene_name": gt_gene_name,
        "Pred_gene_name_list": "; ".join(pred_name_list),
        "gene_name_match": gene_match,

        "GT_disease_list": "; ".join(gt_dis_list),
        "Pred_disease_list": "; ".join(sorted(pred_dis_list)),
        "disease_match": disease_match,

        "GT_phenotype_list": "; ".join(gt_phe_list),
        "Pred_phenotype_list": "; ".join(sorted(pred_phe_list)),
        "phenotype_match": phenotype_match,

        "GT_inconsistency_flag": len(inconsistencies) > 0,
        "GT_inconsistency_details": str(inconsistencies)
    }


In [92]:
import pandas as pd
import numpy as np

def score_and_summarize(all_agg):
    scored = all_agg.copy()

    bool_fields = ["symbol_match", "gene_name_match", "disease_match", "phenotype_match"]

    # Convert True→1, False→0, None→NaN
    for f in bool_fields:
        scored[f + "_score"] = scored[f].map({True: 1, False: 0, None: np.nan})

    summary_rows = []

    for f in bool_fields:
        col = f + "_score"

        true_count  = scored[col].sum(skipna=True)
        valid_count = scored[col].notna().sum()
        false_count = valid_count - true_count
        missing_gt  = len(scored) - valid_count

        match_pct = true_count / valid_count if valid_count > 0 else None

        summary_rows.append({
            "field": f,
            "true_count": int(true_count),
            "false_count": int(false_count),
            "valid_count": int(valid_count),
            "missing_gt": int(missing_gt),
            "match_pct": match_pct
        })

    summary_df = pd.DataFrame(summary_rows)

    # Add % formatted column
    summary_df["match_pct (%)"] = (summary_df["match_pct"] * 100).round(2)

    return scored, summary_df



In [32]:
excel_path = "D:/GeneAgent3/干净数据/20251112_ALL_LLM_results_with_metadata_cleaned.xlsx"
xls = pd.read_excel(excel_path, sheet_name=None)
df = xls["GPT-4o"]

In [33]:
model = "GPT-4o"
# Expected ground truth columns
gt_cols = ["symbol", "gene_name", "disease", "phenotypes"]

# Expected model columns for this sheet
sym_col = f"Symbol ({model_name})"
name_col = f"Gene Name ({model_name})"
dis_col = f"Disease ({model_name})"
phe_col = f"CHD Phenotype ({model_name})"

model_cols = [sym_col, name_col, dis_col, phe_col]

print("GT columns present:", [c for c in gt_cols if c in df.columns])
print("Model columns present:", [c for c in model_cols if c in df.columns])

GT columns present: ['symbol', 'gene_name', 'disease', 'phenotypes']
Model columns present: ['Symbol (GPT-4o)', 'Gene Name (GPT-4o)', 'Disease (GPT-4o)', 'CHD Phenotype (GPT-4o)']


In [34]:
import pandas as pd
import re
aggregated_rows = []
for gid, group in df.groupby("gene_id"):
    aggregated_rows.append(aggregate_gene(group, model, gene_id=gid))

In [36]:
input_path = "D:/GeneAgent3/干净数据/20251112_ALL_LLM_results_with_metadata_full_name.xlsx"
xls = pd.read_excel(input_path, sheet_name=None)
model = "GPT-4o"    # choose your model sheet
df = xls[model]

all_agg = pd.DataFrame([
    aggregate_gene(df_group, model)
    for gene, df_group in df.groupby("symbol")
])

all_agg

,symbol,Pred_symbol_list,symbol_match,gene_name,Pred_gene_name_list,gene_name_match,GT_disease_list,Pred_disease_list,disease_match,GT_phenotype_list,Pred_phenotype_list,phenotype_match,GT_inconsistency_flag,GT_inconsistency_details
0,abl1; abl1,abl1; abl1,True,"ABL proto-oncogene 1, non-receptor tyrosine ki...",abelson murine leukemia viral oncogene homolog...,True,congenital heart defects and skeletal malforma...,not explicitly stated in database; not specifi...,True,atrial septal defect; ventricular septal defect,aortic root dilation; atrial septal defect; co...,True,False,{}
1,actc; actc; actc1,actc; actc; actc1,True,actin alpha cardiac muscle 1,cardiac actin; alpha-cardiac actin; alpha-card...,True,atrial septal defect,atrial septal defect; hypertrophic cardiomyopa...,True,atrial septal defect; ventricular septal defect,apical hypertrophic cardiomyopathy; apical lef...,True,False,{}
2,alk2; alk2,alk2; alk2,False,activin A receptor type 1,activin receptor-like kinase; activin a recept...,True,,congenital heart defects; not explicitly state...,False,atrial septal defect; atrioventricular septal ...,atrioventricular septal defect; atrioventricul...,True,False,{}
3,acvr2b; zic3,acvr2b; zic3,True,activin A receptor type 2B,activin a receptor type 2b; zinc finger protei...,True,"heterotaxy, visceral,","heterotaxy, visceral,; heterotaxy, visceral, x...",True,atrial septal defect; ventricular septal defec...,atrial septal defect; atrioventricular septal ...,True,False,{}
4,adamts10; adamts10,adamts10; adamts10,True,ADAM metallopeptidase with thrombospondin type...,a disintegrin and metalloprotease with thrombo...,True,"weill-marchesani syndrome , recessive",weill-marchesani syndrome; weill-marchesani sy...,True,ventricular septal defect; aortic stenosis; pu...,aortic stenosis with dysplastic valves; brachy...,True,False,{}
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175,acvr2b; nkx2-; zic3; zic3; zic3,acvr2b; nkx2-; zic3; zic3; zic3,True,Zic family member 3,activin a receptor type 2b; nk2 homeobox; zic ...,True,"congenital heart defects, nonsyndromic, , x-li...","congenital heart defect; heterotaxy, visceral,...",True,atrial septal defect; ventricular septal defec...,--; abnormalities of superior and inferior ven...,True,False,{}
176,zmym2,zmym2,True,zinc finger MYM-type containing 2,zinc finger mym-type containing,True,,zmym2-related neurodevelopmental disorder with...,False,atrial septal defect; ventricular septal defec...,atrial septal defect; bicuspid aortic valve; p...,True,False,{}
177,,,False,zinc finger MYND-type containing 10,,False,,,False,heterotaxy,,False,False,{}
178,zmynd8,zmynd8,True,zinc finger MYND-type containing 8,zinc finger mynd domain-containing protein,True,,neurodevelopmental disorder with cardiac malfo...,False,atrial septal defect; ventricular septal defec...,agenesis of the pulmonary artery; atrial septa...,True,False,{}


In [24]:
score_and_summarize(all_agg)

({'symbol_match': {'true_count': 147, 'false_count': 33},
  'gene_name_match': {'true_count': 94, 'false_count': 86},
  'disease_match': {'true_count': 83, 'false_count': 97},
  'phenotype_match': {'true_count': 152, 'false_count': 28}},
 {'total_genes': 180, 'total_true': 476, 'total_false': 244})

In [ ]:
# fuzz match ratio: 80
score_and_summarize(all_agg)

({'symbol_match': {'true_count': 147, 'false_count': 33},
  'gene_name_match': {'true_count': 124, 'false_count': 56},
  'disease_match': {'true_count': 100, 'false_count': 80},
  'phenotype_match': {'true_count': 154, 'false_count': 26}},
 {'total_genes': 180, 'total_true': 525, 'total_false': 195})

In [32]:
# fuzz match ratio: 65
score_and_summarize(all_agg)

({'symbol_match': {'true_count': 147, 'false_count': 33},
  'gene_name_match': {'true_count': 132, 'false_count': 48},
  'disease_match': {'true_count': 108, 'false_count': 72},
  'phenotype_match': {'true_count': 157, 'false_count': 23}},
 {'total_genes': 180, 'total_true': 544, 'total_false': 176})

In [37]:
# fuzz match ratio: 50
score_and_summarize(all_agg)

({'symbol_match': {'true_count': 147, 'false_count': 33},
  'gene_name_match': {'true_count': 138, 'false_count': 42},
  'disease_match': {'true_count': 114, 'false_count': 66},
  'phenotype_match': {'true_count': 159, 'false_count': 21}},
 {'total_genes': 180, 'total_true': 558, 'total_false': 162})

In [51]:
score_and_summarize(pd.DataFrame(aggregated_rows))

({'symbol_match': {'true_count': 147, 'false_count': 33},
  'gene_name_match': {'true_count': 138, 'false_count': 42},
  'disease_match': {'true_count': 114, 'false_count': 66},
  'phenotype_match': {'true_count': 159, 'false_count': 21}},
 {'total_genes': 180, 'total_true': 558, 'total_false': 162})

In [39]:
#all_agg.to_csv(f"D:/GeneAgent3/干净数据/GPT-4o_all_agg_ratio.50.csv", index=False)

In [55]:
score_and_summarize(pd.DataFrame(aggregated_rows)) # 85% ratio

,field,true_count,false_count,valid_count,missing_gt,match_pct,match_pct (%)
0,symbol_match,147,16,163,0,0.901840,90.18
1,gene_name_match,118,45,163,0,0.723926,72.39
2,disease_match,100,28,128,35,0.781250,78.12
3,phenotype_match,152,11,163,0,0.932515,93.25


In [45]:
fuzzy_match(
    "zinc finger MYND-type containing 8",
    "Zinc finger MYND domain-containing protein 8"
)

False

In [34]:
fuzzy_match(
    "catenin beta 1",
    "catenin beta", threshold=99
)

True

In [67]:
fuzz.token_set_ratio("tolloid like 1", "tolloid like") 

100.0

In [51]:
fuzz.token_set_ratio("zinc finger MYND-type containing 8", "Zinc finger MYND domain-containing protein 8") >= 65

True

In [32]:
fuzz.token_set_ratio("catenin beta 1", "catenin beta")

100.0

In [45]:
fuzz.token_set_ratio("ras like without caax 1", "ras-like without caax")

77.27272727272728

In [61]:
fuzzy_match(
    "Ras like without CAAX 1",
    "ras-like without caax", threshold=85
)

False

In [59]:
fuzzy_match(
    "spalt like transcription factor 4",
    "spalt-like transcription factor", threshold=85
)

False

In [ ]:
def summarize_column(col_name):
    print(f"\n=== Column: {col_name} ===")
    if col_name not in df.columns:
        print("  (missing in this sheet)")
        return
    
    col = df[col_name]
    print("  dtype:", col.dtype)
    print("  #rows:", len(col))
    print("  #NaN:", col.isna().sum())
    
    # Show a few unique examples (string-truncated)
    sample = col.dropna().astype(str).head(10).tolist()
    print("  Sample values:")
    for s in sample:
        print("   -", s[:100])

for c in gt_cols + model_cols:
    summarize_column(c)